# Clase: importancia del preprocesamiento despues del EDA

En el EDA observamos la estructura y el comportamiento de los datos. Ahora prepararemos las variables antes de entrenar un modelo y veremos por que cada transformacion debe aprenderse solo con los datos de entrenamiento.

## 1. Cargar el dataset Parquet

El archivo `data/processed/crop.parquet` contiene las observaciones revisadas en el analisis anterior. En la siguiente celda mostramos su tamano, algunas filas y los tipos de datos para confirmar con que vamos a trabajar.

Avanzaremos por cada paso y observaremos las tablas que resultan de cada transformacion.

In [15]:
"""Practica guiada: preprocesamiento y prevencion de data leakage."""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler


def mostrar_tabla(titulo, tabla):
    """Muestra un titulo y una tabla de pandas durante la explicacion."""
    print(titulo)
    display(tabla)


# Buscar la raiz hace que el notebook funcione aunque Jupyter se abra desde otra carpeta.
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data" / "processed" / "crop.parquet").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "No se encontro data/processed/crop.parquet desde el directorio actual."
    )

# El parquet es la salida del EDA; mostramos su estructura antes de transformarlo.
source_path = project_root / "data" / "processed" / "crop.parquet"
df = pd.read_parquet(source_path)
print(f"Dataset cargado desde: {source_path}")
print(f"Filas y columnas: {df.shape}")
mostrar_tabla("Primeras filas del dataset:", df.head())
mostrar_tabla("Tipos de datos:", df.dtypes.rename("tipo").to_frame())

Dataset cargado desde: c:\Users\marco\ml-demo1-profesor\data\processed\crop.parquet
Filas y columnas: (2200, 8)
Primeras filas del dataset:


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


Tipos de datos:


,tipo
N,int64
P,int64
K,int64
temperature,float64
humidity,float64
ph,float64
rainfall,float64
label,str


## 2. Simular valores faltantes

En datos reales pueden faltar mediciones. Para observar el efecto, introduciremos un 5% de valores `NaN` en `N`, `P`, `K` y `temperature`. Usamos una semilla fija para que el ejemplo sea reproducible, y seleccionamos exactamente el 5% de las filas en cada columna.

In [ ]:
# 2. Simular NaN con seleccion aleatoria reproducible (5% exacto por columna).
missing_value_columns = ["N", "P", "K", "temperature"]
target_column = "label"
required_columns = missing_value_columns + [target_column]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas en el parquet: {missing_columns}")

feature_columns = [column for column in df.columns if column != target_column]
df_simulado = df.copy()
rng = np.random.default_rng(42)  # Misma mascara de faltantes en cada ejecucion.
missing_count = int(round(0.05 * len(df_simulado)))
for column in missing_value_columns:
    # Elegir filas sin reemplazo asegura exactamente 5% en cada columna.
    missing_indices = rng.choice(df_simulado.index, size=missing_count, replace=False)
    df_simulado.loc[missing_indices, column] = np.nan

print(f"Se asignaron {missing_count} NaN por cada columna (5% de {len(df_simulado)} filas).")
filas_con_faltantes = df_simulado[
    df_simulado[missing_value_columns].isna().any(axis=1)
]
mostrar_tabla(
    "Filas de ejemplo que contienen valores faltantes:",
    filas_con_faltantes[feature_columns + [target_column]].head(12),
)

Se asignaron 110 NaN por cada columna (5% de 2200 filas).
Filas de ejemplo que contienen valores faltantes:


,N,P,K,temperature,humidity,ph,rainfall,label
13,93.0,56.0,NaN,24.014976,82.056872,6.984354,185.277339,rice
16,NaN,38.0,41.0,NaN,82.788371,6.249051,276.655246,rice
21,76.0,NaN,43.0,25.157455,83.117135,5.070176,231.384316,rice
37,95.0,39.0,NaN,23.863305,83.152508,5.561399,285.249365,rice
41,64.0,NaN,43.0,25.629801,83.528423,5.534878,209.900198,rice
46,76.0,NaN,NaN,24.958779,84.479634,5.206373,196.956001,rice
48,79.0,NaN,39.0,21.666283,80.709606,7.062779,210.814209,rice
52,93.0,56.0,NaN,23.857240,82.225730,7.382763,195.094831,rice
73,78.0,43.0,42.0,NaN,83.003205,7.283737,192.319754,rice
82,61.0,NaN,43.0,26.403232,81.056355,6.349606,223.367188,rice


## 3. Contar los valores faltantes

Antes de imputar, contamos los `NaN` en cada predictor. En la celda siguiente usamos las variables de entrada `X` para ver cuantos valores faltan por columna.

In [3]:
# 3. Crear X y revisar los valores faltantes antes de imputar.
X = df_simulado[feature_columns].copy()
missing_before = X.isna().sum().rename("NaN antes de imputar")
mostrar_tabla("Valores faltantes por predictor:", missing_before.to_frame())
mostrar_tabla("Primeras filas de X:", X.head())

Valores faltantes por predictor:


,NaN antes de imputar
N,110
P,110
K,110
temperature,110
humidity,0
ph,0
rainfall,0


Primeras filas de X:


,N,P,K,temperature,humidity,ph,rainfall
0,90.0,42.0,43.0,20.879744,82.002744,6.502985,202.935536
1,85.0,58.0,41.0,21.770462,80.319644,7.038096,226.655537
2,60.0,55.0,44.0,23.004459,82.320763,7.840207,263.964248
3,74.0,35.0,40.0,26.491096,80.158363,6.980401,242.864034
4,78.0,42.0,42.0,20.130175,81.604873,7.628473,262.717340


## 5. Separar X e y

`X` contiene todas las variables predictoras y `y` contiene unicamente `label`, que es la categoria que queremos predecir. Separarlas permite enviar cada parte correctamente a las funciones de scikit-learn.

In [13]:
# 5. Definir la variable objetivo y mostrar algunos ejemplos.
y = df_simulado[target_column].copy()
mostrar_tabla("Primeras etiquetas de y (label):", y.head().to_frame())
print(f"Numero de predictores en X: {X.shape[1]}")

Primeras etiquetas de y (label):


,label
0,rice
1,rice
2,rice
3,rice
4,rice


Numero de predictores en X: 7


## 6. Separar entrenamiento y prueba

Reservamos el 20% para evaluar el modelo en datos que no vio durante el entrenamiento. `stratify=y` conserva la proporcion de cada cultivo en ambos conjuntos, y `random_state=42` hace que la particion se pueda repetir.

Aunque la lista de pasos inicial pone la imputacion antes del split, en un proceso correcto primero separamos los datos y despues aprendemos la mediana solo con train. Asi test no influye en ninguna transformacion.

In [ ]:
# 6. Separar antes de aprender medianas o escalas para que test no filtre informacion.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,  # Conserva la proporcion de cultivos en train y test.
    random_state=42,
)
print(f"Filas en train: {X_train.shape[0]} ({X_train.shape[0] / len(X):.0%})")
print(f"Filas en test:  {X_test.shape[0]} ({X_test.shape[0] / len(X):.0%})")
mostrar_tabla(
    "Distribucion de clases en train (proporcion):",
    y_train.value_counts(normalize=True).sort_index().rename("proporcion").to_frame(),
)
mostrar_tabla("Ejemplo de X_train (aun puede contener NaN):", X_train.head())
mostrar_tabla("Ejemplo de X_test (aun puede contener NaN):", X_test.head())

Filas en train: 1760 (80%)
Filas en test:  440 (20%)
Distribucion de clases en train (proporcion):


,proporcion
label,
apple,0.045455
banana,0.045455
blackgram,0.045455
chickpea,0.045455
coconut,0.045455
coffee,0.045455
cotton,0.045455
grapes,0.045455
jute,0.045455


Ejemplo de X_train (aun puede contener NaN):


,N,P,K,temperature,humidity,ph,rainfall
1607,0.0,18.0,14.0,29.771494,92.007200,7.207991,114.416179
1212,9.0,122.0,201.0,29.587484,80.919344,5.570291,68.064173
362,11.0,71.0,24.0,21.140114,22.718235,5.606620,141.605672
566,38.0,38.0,18.0,26.310518,61.187491,6.294130,35.734038
1671,18.0,12.0,8.0,12.590940,91.816688,6.206053,119.391672


Ejemplo de X_test (aun puede contener NaN):


,N,P,K,temperature,humidity,ph,rainfall
1609,13.0,23.0,6.0,23.961476,90.264080,7.365338,102.695870
1072,98.0,79.0,50.0,25.341198,84.473213,6.435917,91.064934
1912,140.0,38.0,NaN,24.147295,75.882986,6.021440,69.915635
100,71.0,54.0,16.0,22.613600,63.690706,5.749914,87.759539
1645,40.0,22.0,6.0,24.536101,91.909972,6.488221,115.978799


## 4. Imputar con la mediana

`SimpleImputer(strategy="median")` reemplaza cada valor faltante por la mediana de su columna. Aunque este paso corresponde al punto 4 de la lista, lo ejecutamos despues del split para evitar leakage: el imputador aprende las medianas de `X_train` y aplica esos mismos valores a train y test.

In [ ]:
# 4. fit aprende las medianas unicamente de train; transform aplica esas medianas a test.
imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), columns=feature_columns, index=X_train.index
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test), columns=feature_columns, index=X_test.index
)
X_imputed = pd.concat([X_train_imputed, X_test_imputed]).loc[X.index]

medianas = pd.Series(imputer.statistics_, index=feature_columns, name="mediana aprendida en train")
mostrar_tabla("Mediana calculada para cada predictor:", medianas.to_frame())
missing_after = X_imputed.isna().sum().rename("NaN despues de imputar")
mostrar_tabla(
    "Comparacion de faltantes antes y despues de imputar:",
    pd.concat([missing_before, missing_after], axis=1),
)
mostrar_tabla("Muestras despues de imputar:", X_imputed.head(10))

Mediana calculada para cada predictor:


,mediana aprendida en train
N,37.000000
P,51.000000
K,32.000000
temperature,25.549633
humidity,80.542322
ph,6.431478
rainfall,94.772563


Comparacion de faltantes antes y despues de imputar:


,NaN antes de imputar,NaN despues de imputar
N,110,0
P,110,0
K,110,0
temperature,110,0
humidity,0,0
ph,0,0
rainfall,0,0


Muestras despues de imputar:


,N,P,K,temperature,humidity,ph,rainfall
0,90.0,42.0,43.0,20.879744,82.002744,6.502985,202.935536
1,85.0,58.0,41.0,21.770462,80.319644,7.038096,226.655537
2,60.0,55.0,44.0,23.004459,82.320763,7.840207,263.964248
3,74.0,35.0,40.0,26.491096,80.158363,6.980401,242.864034
4,78.0,42.0,42.0,20.130175,81.604873,7.628473,262.717340
5,69.0,37.0,42.0,23.058049,83.370118,7.073454,251.055000
6,69.0,55.0,38.0,22.708838,82.639414,5.700806,271.324860
7,94.0,53.0,40.0,20.277744,82.894086,5.718627,241.974195
8,89.0,54.0,38.0,24.515881,83.535216,6.685346,230.446236
9,68.0,58.0,38.0,23.223974,83.033227,6.336254,221.209196


## 7. Escalar con RobustScaler sin leakage

`RobustScaler` centra cada variable con su mediana y la escala usando el rango intercuartil. En el flujo correcto, el scaler se ajusta con train y solo transforma test; por eso test no participa en el calculo de medianas ni rangos.

In [ ]:
# 7. fit_transform solo en train; en test se usa transform con la misma escala aprendida.
scaler_correcto = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler_correcto.fit_transform(X_train_imputed),
    columns=feature_columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler_correcto.transform(X_test_imputed),
    columns=feature_columns,
    index=X_test.index,
)

mostrar_tabla(
    "Medianas de train escalado (deben estar cerca de 0):",
    X_train_scaled.describe().loc[["mean", "50%", "min", "max"]],
)
mostrar_tabla("Muestras de train escalado:", X_train_scaled.head())
mostrar_tabla("Muestras de test escalado:", X_test_scaled.head())

Medianas de train escalado (deben estar cerca de 0):


,N,P,K,temperature,humidity,ph,rainfall
mean,0.210572,0.057491,0.587719,0.025323,-0.307124,4.340272e-02,0.147002
50%,0.000000,0.000000,0.000000,0.000000,0.000000,-4.551210e-16,0.000000
min,-0.606557,-1.210526,-1.038462,-3.005583,-2.230822,-2.999292e+00,-1.262888
max,1.688525,2.473684,6.653846,3.387631,0.654245,3.590483e+00,3.451667


Muestras de train escalado:


,N,P,K,temperature,humidity,ph,rainfall
1607,-0.606557,-0.868421,-0.692308,0.789044,0.385855,0.795767,0.332715
1212,-0.459016,1.868421,6.500000,0.754654,0.012689,-0.882540,-0.452375
362,-0.426230,0.526316,-0.307692,-0.824117,-1.946091,-0.845309,0.793239
566,0.016393,-0.342105,-0.538462,0.142206,-0.651394,-0.140753,-0.999970
1671,-0.311475,-1.026316,-0.923077,-2.421914,0.379443,-0.231014,0.416988


Muestras de test escalado:


,N,P,K,temperature,humidity,ph,rainfall
1609,-0.393443,-0.736842,-1.000000,-0.296818,0.327189,0.957015,0.134202
1072,1.000000,0.736842,0.692308,-0.038955,0.132296,0.004550,-0.062798
1912,1.688525,-0.342105,0.000000,-0.262090,-0.156812,-0.420205,-0.421016
100,0.557377,0.078947,-0.615385,-0.548730,-0.567147,-0.698462,-0.118784
1645,0.049180,-0.763158,-1.000000,-0.189424,0.382582,0.058150,0.359182


## 8. Entrenar Random Forest y medir accuracy

Entrenamos el `RandomForestClassifier` con los datos escalados de train y medimos su accuracy sobre test. El conjunto de prueba se mantiene reservado hasta esta evaluacion.

In [ ]:
# 8. Entrenar solamente con train y reservar test para medir generalizacion.
modelo_correcto = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
modelo_correcto.fit(X_train_scaled, y_train)
predicciones_correctas = modelo_correcto.predict(X_test_scaled)
accuracy_correcto = accuracy_score(y_test, predicciones_correctas)
print(f"Accuracy con preprocesamiento correcto: {accuracy_correcto:.4f}")
mostrar_tabla(
    "Ejemplos de etiquetas reales y predichas:",
    pd.DataFrame(
        {"real": y_test.head(10), "prediccion": predicciones_correctas[:10]},
        index=y_test.head(10).index,
    ),
)

Accuracy con preprocesamiento correcto: 0.9932
Ejemplos de etiquetas reales y predichas:


,real,prediccion
1609,orange,orange
1072,banana,banana
1912,cotton,cotton
100,maize,maize
1645,orange,orange
221,chickpea,chickpea
28,rice,rice
701,blackgram,blackgram
1097,banana,banana
1638,orange,pomegranate


## 9. Repetir con data leakage intencional

En este flujo incorrecto ajustamos `RobustScaler` con todas las filas imputadas, incluidas las que pertenecen a test, antes de extraer las particiones para este escalado. El scaler conoce estadisticas del conjunto de prueba. Mantenemos los mismos indices y modelo para aislar el efecto de esa fuga.

In [ ]:
# 9. FUGA INTENCIONAL: ajustar el scaler con todas las filas revela tambien estadisticas de test.
scaler_con_fuga = RobustScaler()
X_all_scaled = pd.DataFrame(
    scaler_con_fuga.fit_transform(X_imputed),
    columns=feature_columns,
    index=X.index,
)
# Seleccionar los mismos indices permite comparar unicamente el efecto de la fuga.
X_train_leak = X_all_scaled.loc[X_train.index]
X_test_leak = X_all_scaled.loc[X_test.index]

mostrar_tabla("Muestras escaladas con fuga (train):", X_train_leak.head())
mostrar_tabla("Muestras escaladas con fuga (test):", X_test_leak.head())

modelo_con_fuga = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
modelo_con_fuga.fit(X_train_leak, y_train)
predicciones_con_fuga = modelo_con_fuga.predict(X_test_leak)
accuracy_con_fuga = accuracy_score(y_test, predicciones_con_fuga)
print(f"Accuracy escalando todo antes de separar (con fuga): {accuracy_con_fuga:.4f}")

Muestras escaladas con fuga (train):


,N,P,K,temperature,humidity,ph,rainfall
1607,-0.606557,-0.868421,-0.666667,0.784671,0.388524,0.822466,0.327360
1212,-0.459016,1.868421,6.259259,0.750471,0.015030,-0.897899,-0.448850
362,-0.426230,0.526316,-0.296296,-0.819549,-1.945473,-0.859735,0.782674
566,0.016393,-0.342105,-0.518519,0.141417,-0.649637,-0.137523,-0.990250
1671,-0.311475,-1.026316,-0.888889,-2.408490,0.382107,-0.230046,0.410679


Muestras escaladas con fuga (test):


,N,P,K,temperature,humidity,ph,rainfall
1609,-0.393443,-0.736842,-0.962963,-0.295173,0.329807,0.987755,0.131092
1072,1.000000,0.736842,0.666667,-0.038740,0.134742,0.011421,-0.063680
1912,1.688525,-0.342105,0.000000,-0.260637,-0.154619,-0.423978,-0.417846
100,0.557377,0.078947,-0.592593,-0.545688,-0.565316,-0.709208,-0.119032
1645,0.049180,-0.763158,-0.962963,-0.188374,0.385249,0.066365,0.353527


Accuracy escalando todo antes de separar (con fuga): 0.9932


## 10. Comparar las accuracies

Las dos mediciones usan el mismo conjunto de prueba. Puede que el resultado del bosque aleatorio sea igual en ambos casos: los arboles suelen depender del orden de los valores y no de su escala. Eso no vuelve correcto el leakage; otros modelos y datos pueden verse afectados.

In [ ]:
# 10. Comparar accuracy en las mismas filas de test; la escala puede no cambiar un arbol.
comparacion = pd.DataFrame(
    {
        "proceso": [
            "Correcto: scaler ajustado solo en train",
            "Incorrecto: scaler ajustado en train y test",
        ],
        "accuracy_test": [accuracy_correcto, accuracy_con_fuga],
    }
)
mostrar_tabla("Comparacion de accuracy:", comparacion)

Comparacion de accuracy:


,proceso,accuracy_test
0,Correcto: scaler ajustado solo en train,0.993182
1,Incorrecto: scaler ajustado en train y test,0.993182


## 11-12. Guardar train procesado y scaler

Guardamos solo el conjunto de entrenamiento, con predictores imputados y escalados por el `RobustScaler` correcto, junto con `label`. El scaler que se guarda es el ajustado unicamente con train; se necesitara para preparar futuros datos con las mismas transformaciones.

In [ ]:
# 11. Guardar unicamente train ya imputado y escalado, junto con la etiqueta.
processed_dir = project_root / "data" / "processed"
models_dir = project_root / "models"
processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

train_processed = X_train_scaled.copy()
train_processed[target_column] = y_train
train_path = processed_dir / "crop_train.parquet"
train_processed.to_parquet(train_path, index=False)

# 12. Persistir el scaler ajustado solo con train, nunca el scaler con fuga.
scaler_path = models_dir / "scaler.pkl"
joblib.dump(scaler_correcto, scaler_path)

print(f"Dataset de train guardado en: {train_path}")
print(f"Scaler guardado en: {scaler_path}")
print(f"Forma del dataset guardado: {train_processed.shape}")
mostrar_tabla("Vista de las filas guardadas:", train_processed.head())

Dataset de train guardado en: c:\Users\marco\ml-demo1-profesor\data\processed\crop_train.parquet
Scaler guardado en: c:\Users\marco\ml-demo1-profesor\models\scaler.pkl
Forma del dataset guardado: (1760, 8)
Vista de las filas guardadas:


,N,P,K,temperature,humidity,ph,rainfall,label
1607,-0.606557,-0.868421,-0.692308,0.789044,0.385855,0.795767,0.332715,orange
1212,-0.459016,1.868421,6.500000,0.754654,0.012689,-0.882540,-0.452375,grapes
362,-0.426230,0.526316,-0.307692,-0.824117,-1.946091,-0.845309,0.793239,kidneybeans
566,0.016393,-0.342105,-0.538462,0.142206,-0.651394,-0.140753,-0.999970,mothbeans
1671,-0.311475,-1.026316,-0.923077,-2.421914,0.379443,-0.231014,0.416988,orange


## Funciones reutilizables

El ejemplo anterior muestra cada transformacion por separado para aprender el proceso. En un proyecto, estas operaciones suelen guardarse como funciones. La regla sigue siendo la misma: primero dividir; luego ajustar imputador y scaler solo con train, y usar `transform` para test.

Estas funciones devuelven tambien los objetos ajustados (`imputer` y `scaler`) para reutilizarlos al preparar datos nuevos.

In [16]:
def load_raw_data(path: str | Path) -> pd.DataFrame:
    """Carga datos desde un archivo Parquet."""
    return pd.read_parquet(path)


def split_data(
    df: pd.DataFrame,
    target: str,
    test_size: float = 0.2,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Separa predictores y objetivo en train/test de forma estratificada."""
    if target not in df.columns:
        raise ValueError(f"No existe la columna objetivo {target!r}.")

    X = df.drop(columns=target)
    y = df[target]
    return train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )


def handle_missing(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    strategy: str = "median",
) -> tuple[pd.DataFrame, pd.DataFrame, SimpleImputer]:
    """Imputa train/test; aprende la estrategia solo desde train."""
    if not X_train.columns.equals(X_test.columns):
        raise ValueError("Train y test deben tener las mismas columnas y en el mismo orden.")

    imputer = SimpleImputer(strategy=strategy)
    train_clean = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    test_clean = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )
    return train_clean, test_clean, imputer


def scale_features(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    scaler_type: str = "robust",
) -> tuple[pd.DataFrame, pd.DataFrame, RobustScaler | StandardScaler]:
    """Ajusta el scaler en train y transforma train/test con la misma escala."""
    if not X_train.columns.equals(X_test.columns):
        raise ValueError("Train y test deben tener las mismas columnas y en el mismo orden.")

    scaler_classes = {"robust": RobustScaler, "standard": StandardScaler}
    if scaler_type not in scaler_classes:
        raise ValueError("scaler_type debe ser 'robust' o 'standard'.")

    scaler = scaler_classes[scaler_type]()
    train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )
    return train_scaled, test_scaled, scaler

In [17]:
# Ejemplo de uso modular: dividir primero y ajustar cada transformador solo con train.
df_recargado = load_raw_data(source_path)
print(f"Dataset recargado por la funcion: {df_recargado.shape}")

X_train_demo, X_test_demo, y_train_demo, y_test_demo = split_data(
    df_simulado,
    target=target_column,
)
X_train_limpio, X_test_limpio, imputer_demo = handle_missing(
    X_train_demo,
    X_test_demo,
    strategy="median",
)
X_train_escalado, X_test_escalado, scaler_demo = scale_features(
    X_train_limpio,
    X_test_limpio,
    scaler_type="robust",
)

print(f"Train: {X_train_escalado.shape}; test: {X_test_escalado.shape}")
print(f"Filas de prueba reservadas para evaluacion: {len(y_test_demo)}")
mostrar_tabla("Resultado de las funciones (train):", X_train_escalado.head())
mostrar_tabla("Resultado de las funciones (test):", X_test_escalado.head())
print(f"Imputador guardado para reutilizar: {type(imputer_demo).__name__}")
print(f"Scaler guardado para reutilizar: {type(scaler_demo).__name__}")

Dataset recargado por la funcion: (2200, 8)
Train: (1760, 7); test: (440, 7)
Filas de prueba reservadas para evaluacion: 440
Resultado de las funciones (train):


,N,P,K,temperature,humidity,ph,rainfall
1607,-0.606557,-0.868421,-0.692308,0.789044,0.385855,0.795767,0.332715
1212,-0.459016,1.868421,6.500000,0.754654,0.012689,-0.882540,-0.452375
362,-0.426230,0.526316,-0.307692,-0.824117,-1.946091,-0.845309,0.793239
566,0.016393,-0.342105,-0.538462,0.142206,-0.651394,-0.140753,-0.999970
1671,-0.311475,-1.026316,-0.923077,-2.421914,0.379443,-0.231014,0.416988


Resultado de las funciones (test):


,N,P,K,temperature,humidity,ph,rainfall
1609,-0.393443,-0.736842,-1.000000,-0.296818,0.327189,0.957015,0.134202
1072,1.000000,0.736842,0.692308,-0.038955,0.132296,0.004550,-0.062798
1912,1.688525,-0.342105,0.000000,-0.262090,-0.156812,-0.420205,-0.421016
100,0.557377,0.078947,-0.615385,-0.548730,-0.567147,-0.698462,-0.118784
1645,0.049180,-0.763158,-1.000000,-0.189424,0.382582,0.058150,0.359182


Imputador guardado para reutilizar: SimpleImputer
Scaler guardado para reutilizar: RobustScaler


## 13. Cierre

Revisa el conteo de faltantes, la particion, las medianas aprendidas, los datos escalados, las predicciones y los archivos que acabamos de guardar.

In [12]:
# 13. Mensaje final solicitado.
print("Preprocesamiento completado. Revisa data/processed/ y models/")

Preprocesamiento completado. Revisa data/processed/ y models/
